In [ ]:
# GitHub Copilot
# New Jupyter cell (index 0)
# Demonstration: selectively unzip certain files from large ZIPs using the zipfile stdlib.
# - list contents with metadata
# - filter by glob / regex / size / extension
# - extract selected members safely and streaming (no full-file memory load)
# - example usage at the bottom (replace ZIP_PATH with your archive)

import zipfile
import fnmatch
import re
from pathlib import Path
import shutil
import os
from typing import Iterable, List, Optional


def list_zip_contents(zip_path: str):
    """Return list of zipinfo-like dicts for members in the archive."""
    with zipfile.ZipFile(zip_path, "r") as z:
        out = []
        for info in z.infolist():
            out.append({
                "name": info.filename,
                "is_dir": info.is_dir(),
                "compressed_size": info.compress_size,
                "uncompressed_size": info.file_size,
                "datetime": info.date_time,
            })
        return out


def filter_members(members: Iterable[str],
                   patterns: Optional[List[str]] = None,
                   regex: Optional[str] = None,
                   min_size: Optional[int] = None,
                   max_size: Optional[int] = None,
                   extensions: Optional[List[str]] = None):
    """Filter a sequence of member names (strings) by various criteria.
    - patterns: list of glob-style patterns (fnmatch), applied to the whole path string
    - regex: full regex applied to the path
    - min_size / max_size: not applied here (size-aware filtering is done using ZipInfo)
    - extensions: list like ['.csv', '.txt'] (case-insensitive)
    """
    results = list(members)
    if patterns:
        results = [
            m for p in patterns
            for m in results
            if fnmatch.fnmatch(m, p)
        ]
        # remove duplicates while preserving order
        seen = set()
        results = [x for x in results if not (x in seen or seen.add(x))]
    if regex:
        cre = re.compile(regex)
        results = [m for m in results if cre.search(m)]
    if extensions:
        extset = {e.lower() if e.startswith('.') else f".{e.lower()}" for e in extensions}
        results = [m for m in results if Path(m).suffix.lower() in extset]
    return results


def _safe_extract_path(dest_dir: Path, member_path: str) -> Path:
    """Prevent ZipSlip by ensuring final path is inside dest_dir."""
    dest_dir = dest_dir.resolve()
    target = (dest_dir / Path(member_path)).resolve()
    if not str(target).startswith(str(dest_dir) + os.sep) and target != dest_dir:
        # not safe
        raise RuntimeError(f"Unsafe path in zip file: {member_path}")
    return target


def extract_selected(zip_path: str,
                     members: Iterable[str],
                     dest_dir: str,
                     overwrite: bool = False):
    """Extract given members to dest_dir safely using streaming (no full file in memory)."""
    dest = Path(dest_dir)
    dest.mkdir(parents=True, exist_ok=True)
    extracted = []
    with zipfile.ZipFile(zip_path, "r") as z:
        for m in members:
            try:
                info = z.getinfo(m)
            except KeyError:
                # member not in archive
                continue
            if info.is_dir():
                (dest / m).mkdir(parents=True, exist_ok=True)
                extracted.append(m)
                continue
            out_path = _safe_extract_path(dest, m)
            out_path.parent.mkdir(parents=True, exist_ok=True)
            if out_path.exists() and not overwrite:
                # skip existing
                continue
            # open compressed member as file-like and stream to disk
            with z.open(info, "r") as src, open(out_path, "wb") as dst:
                shutil.copyfileobj(src, dst)  # streaming copy, low memory
            # preserve modified time? you can set file mtime if desired using info.date_time
            extracted.append(m)
    return extracted


def extract_by_criteria(zip_path: str,
                        dest_dir: str,
                        patterns: Optional[List[str]] = None,
                        regex: Optional[str] = None,
                        extensions: Optional[List[str]] = None,
                        min_uncompressed_size: Optional[int] = None,
                        max_uncompressed_size: Optional[int] = None,
                        overwrite: bool = False):
    """Combine listing + filtering by ZipInfo (size-aware) then extract selected members."""
    with zipfile.ZipFile(zip_path, "r") as z:
        infos = z.infolist()
        # initial name list
        names = [info.filename for info in infos if not info.is_dir()]
        # apply name-based filters
        names = filter_members(names, patterns=patterns, regex=regex, extensions=extensions)
        # apply size filters using the ZipInfo mapping
        selected = []
        info_map = {info.filename: info for info in infos}
        for name in names:
            info = info_map.get(name)
            if info is None:
                continue
            if min_uncompressed_size and info.file_size < min_uncompressed_size:
                continue
            if max_uncompressed_size and info.file_size > max_uncompressed_size:
                continue
            selected.append(name)
    # extract the selected names
    return extract_selected(zip_path, selected, dest_dir, overwrite=overwrite)


def stream_process_member(zip_path: str, member: str, process_fn):
    """Open a member and pass the file-like object to process_fn for streaming processing.
    process_fn(fileobj) -> any
    Example: read first N bytes, iterate lines, decompress on the fly, etc.
    """
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(member, "r") as f:
            return process_fn(f)


# ----------------------
# Example usage (replace ZIP_PATH with your large archive)
# ----------------------
if __name__ == "__main__":
    ZIP_PATH = "large_archive.zip"  # <- update with your archive path
    DEST = "selected_out"

    # 1) Inspect archive contents
    contents = list_zip_contents(ZIP_PATH)
    print(f"Archive has {len(contents)} entries; sample 10:")
    for it in contents[:10]:
        print(it)

    # 2) Extract all CSVs under data/ directory larger than 10 KB
    extracted = extract_by_criteria(
        ZIP_PATH,
        dest_dir=DEST,
        patterns=["data/*.csv", "data/**/*.csv"],
        extensions=[".csv"],
        min_uncompressed_size=10 * 1024,
        overwrite=False
    )
    print(f"Extracted {len(extracted)} CSV files to {DEST}")

    # 3) Stream a single file without extracting to disk (process first 200 bytes)
    # Replace 'path/inside.zip/file.txt' with an actual member name from the archive
    member_name = None
    # pick one text-like file from contents as an example
    for c in contents:
        if c["name"].lower().endswith((".txt", ".csv", ".log")):
            member_name = c["name"]
            break

    if member_name:
        head = stream_process_member(ZIP_PATH, member_name, lambda f: f.read(200))
        print(f"First 200 bytes of {member_name!r}:", head[:200])
    else:
        print("No text-like member found to demonstrate streaming.")